# 1. CARGA DE DATOS Y LIMPIEZA INICIAL
Carga el archivo CSV con la matriz de datos urbanos y espaciales de Bogotá, mostrando una vista previa de su estructura.

In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Cargamos la matriz final resultante del proceso ETL
df = pd.read_csv('../data/matriz_barrios_unificada.csv')

# Eliminamos los registros que no tienen valor de referencia para 2026 (nuestra variable objetivo)
df_clean = df.dropna(subset=['v_ref_2026']).copy()

In [3]:
df_clean.head()

,localidad,barrio,estrato_predominante,area_km2,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,densidad_airbnb_km2,geometry
0,USME,EL BOSQUE,1.0,0.055879,290000.0,650000.0,660000.0,0.084056,0,0.0,0.000000,POLYGON ((-74.10014659299998 4.505445796000060...
1,SUBA,BILBAO,2.0,0.295314,750000.0,1790000.0,2050000.0,0.090886,0,0.0,0.000000,"POLYGON ((-74.113150507 4.751871279999989, -74..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,3.0,0.476473,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,14.691286,POLYGON ((-74.15600688799998 4.616521243000022...
3,CHAPINERO,CONCEPCION NORTE,3.0,0.140259,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,0.000000,POLYGON ((-74.06568096400002 4.656363177999992...
4,PUENTE ARANDA,COLON,3.0,0.285947,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,55.954334,POLYGON ((-74.11963554699997 4.619638797999983...


In [8]:
print(f"Registros originales: {len(df)} | Registros tras filtrar nans en v_ref_2026: {len(df_clean)}")

Registros originales: 1098 | Registros tras filtrar nans en v_ref_2026: 1090


# 2. TRANSFORMACIONES LOGARÍTMICAS

In [4]:
# Aplicamos logaritmo natural al precio para estabilizar la varianza (heterocedasticidad)
df_clean['log_v_ref_2026'] = np.log(df_clean['v_ref_2026'])

# Aplicamos log1p (logaritmo de x + 1) a la densidad porque tiene valores en 0
df_clean['log_densidad_airbnb'] = np.log1p(df_clean['densidad_airbnb_km2'])

# 3. TRATAMIENTO DE DATOS ATÍPICOS (Rango Intercuartílico - IQR)

In [5]:
# Calculamos los límites para aislar los barrios con precios extremadamente altos o bajos
Q1 = df_clean['log_v_ref_2026'].quantile(0.25)
Q3 = df_clean['log_v_ref_2026'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Filtramos la base para mantener solo los datos dentro de una distribución normal
df_model = df_clean[(df_clean['log_v_ref_2026'] >= limite_inferior) & 
                    (df_clean['log_v_ref_2026'] <= limite_superior)].copy()

print(f"Registros originales (sin NaNs): {len(df_clean)} | Registros tras filtrar atípicos: {len(df_model)}")

Registros originales (sin NaNs): 1090 | Registros tras filtrar atípicos: 1058


# 4. PREPARACIÓN DE VARIABLES Y PARTICIÓN (TRAIN/TEST)
Divide la matriz supervisada en conjuntos de entrenamiento (80%) y prueba (20%) para evaluar de manera objetiva el rendimiento del modelo.

In [9]:
# Definimos las variables predictoras (X) y la variable objetivo (y)
variables_predictoras = [
    'log_densidad_airbnb',          # Efecto turístico transformado
    'estrato_predominante',         # Control socioeconómico
    'num_estaciones_transmilenio',  # Control de conectividad
    'cagr_15_25'                    # Inercia histórica del mercado
]

X = df_model[variables_predictoras]
y = df_model['log_v_ref_2026']

# Agregamos la constante (intercepto) requerida por statsmodels para el modelo OLS
X = sm.add_constant(X)

# Dividimos en 80% entrenamiento y 20% prueba (fijando semilla para reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. ENTRENAMIENTO DEL MODELO OLS

In [10]:
# Ajustamos el modelo con los datos de entrenamiento
modelo_ols = sm.OLS(y_train, X_train).fit()

# Imprimimos el resumen estadístico exhaustivo
print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:         log_v_ref_2026   R-squared:                       0.463
Model:                            OLS   Adj. R-squared:                  0.461
Method:                 Least Squares   F-statistic:                     181.5
Date:                Mon, 02 Nov 2026   Prob (F-statistic):          4.39e-112
Time:                        16:41:18   Log-Likelihood:                -834.53
No. Observations:                 846   AIC:                             1679.
Df Residuals:                     841   BIC:                             1703.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const             

# 6. EVALUACIÓN DEL MODELO EN EL CONJUNTO DE PRUEBA (TEST)

In [13]:
# 1. Generar las predicciones con los datos que el modelo nunca ha visto
y_pred_log = modelo_ols.predict(X_test)

# 2. Evaluar en la escala logarítmica (escala matemática del entrenamiento)
r2_test = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

print("--- MÉTRICAS EN ESCALA LOGARÍTMICA (ESTADÍSTICAS) ---")
print(f"R-cuadrado en Test: {r2_test:.4f}")
print(f"MAE (Log): {mae_log:.4f}")
print(f"RMSE (Log): {rmse_log:.4f}\n")

# 3. Evaluar en la escala real (Pesos Colombianos) para interpretación de negocio
# Revertimos el logaritmo natural usando la función exponencial de numpy (np.exp)
y_test_real = np.exp(y_test)
y_pred_real = np.exp(y_pred_log)

mae_real = mean_absolute_error(y_test_real, y_pred_real)
rmse_real = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print("--- MÉTRICAS EN ESCALA REAL DE NEGOCIO ($ COP / m²) ---")
print(f"Error Absoluto Medio (MAE): ${mae_real:,.0f}")
print(f"Raíz del Error Cuadrático Medio (RMSE): ${rmse_real:,.0f}")

--- MÉTRICAS EN ESCALA LOGARÍTMICA (ESTADÍSTICAS) ---
R-cuadrado en Test: 0.4412
MAE (Log): 0.5056
RMSE (Log): 0.6731

--- MÉTRICAS EN ESCALA REAL DE NEGOCIO ($ COP / m²) ---
Error Absoluto Medio (MAE): $1,038,950
Raíz del Error Cuadrático Medio (RMSE): $1,564,948
